# PhysAR-IDS -- NB14 (v6): ablations -- which features and which defence components carry the result

Two ablations, both on the v6 pipeline, 5 seeds, paired statistics. Nothing here is a new claim about the data; it tests which parts of the system the dissertation's claims actually rest on.

**A. Feature ablation (tuned XGBoost, refitted per feature set).** Nested sets built from the 93 features: physical only (33), cyber only (60), legacy cyber (the original per-second features), legacy + 2 s windows, physical + legacy cyber, all 93, and all 93 without the 4 derived physical features (voltage / current imbalance, power factor, frequency deviation).
For each set: clean weighted / macro F1 and per-class recall, and the **white-box tree-greedy attack of NB3c run against that set's own trees** at 0.3 sigma and 1.0 sigma, with two attackers: *cyber attacker* (may move only the cyber features the model uses; the threat model of the rest of the dissertation) and *any-feature attacker* (may also move physical channels; 0.3 sigma). A model with no cyber features cannot be attacked by the cyber attacker, so that cell is empty rather than 0.
NB1 already reports clean F1 for a few groups with the NB1 hyper-parameters; here every set uses the NB2 tuned parameters and is attacked, which NB1 did not do.

**B. Defence-component ablation.** Classifier: `base` (tuned XGBoost, NB2), `rob` (iterative white-box adversarial training, NB5b, file `xgb_robust`), `dfn` (PGD + PPO adversarial training, NB5, file `xgb_defended`). Each with and without the autoencoder gate (NB6 gate, its own 95th-percentile threshold). Conditions: clean, grey-box PGD (NB3 surrogate), white-box tree-greedy at 0.3 and 1.0 sigma (cyber features) **against the classifier's own trees**.
Measured: evasion of the classifier alone, of the gate alone, of the two together (a row evades when the classifier says benign *and* the gate does not flag it), the false-alarm rate of each, and the ratio observed / (product of the single-layer evasions), which tests whether layers behave independently.
**The attack here does not see the gate.** A gate-aware attacker is NB6's job; this table measures what the layers add against an attacker that targets the classifier only, so "the layers compound" can be checked rather than assumed.

**Definitions.** Evasion = share of attack rows classified benign after the attack, including the share already missed on clean data (printed as the clean value). Attack rows are a fixed sample per attack class from the test split (`WB_PER_CLASS` per class); rows within a burst are identical copies, so the independent unit is the burst (test has 422) -- seed std measures model variance only.

**Not repeated here:** the v4 architecture x SCL x adversarial-training ablation (Table 6.10). It depended on the SCL encoders and a transfer attack that no v6 notebook reproduces; either rerun it separately or drop it from the dissertation. Gate-input ablations (physical / cyber / weighted) are in NB7.

**Needs:** `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`, `xgb_robust_v6_seed*.json`, `xgb_defended_v6_seed*.json`, `ae_gate_v6_seed*.pt`, `surrogate_mlp_v6_seed*.pt`. **Runtime:** roughly 2-3 h on the local GPU (7 fits + about 20 white-box searches per seed). Results are checkpointed per seed.

## 0. Setup

In [ ]:
import os, sys, random, time, json, warnings, math
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
ALPHAS = [0.1, 0.05, 0.01]; ATK_ALPHAS = [0.1, 0.05]
CAL_FRAC = 0.15; GAP_FRAC = 0.02
N_BOOT = 2000
PGD_EPS = 0.1; PGD_ITERS = 10
WB_PER_CLASS = 300; WB_BUDGET = 0.3                  # white-box: sample per attack class, sigma on the 60 cyber features
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
N_BINS = 15

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"}')
print('Working directory:', os.getcwd())

In [ ]:
WB_PER_CLASS = 60 if SMOKE_TEST else 200               # attack rows per attack class (fixed sample)
WB_BUDGETS = [0.3, 1.0]                                 # sigma, L-inf
PGD_EPS = 0.1
class AE(nn.Module):                                    # NB6 gate architecture
    def __init__(self, n_in, bottleneck=12, hidden=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_in, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_in))
    def forward(self, x): return self.decoder(self.encoder(x))

## 1. Load v6 data and define the feature sets

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']).astype(np.int64); y_test = np.asarray(data['y_test']).astype(np.int64)
plc_tr, plc_te = np.asarray(data['PLCID_train']), np.asarray(data['PLCID_test']); ts_tr, ts_te = np.asarray(data['timestamp_train']), np.asarray(data['timestamp_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; PHYS_IDX = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']; atk_names = class_names[1:]
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
for k_ in ('n_estimators', 'max_depth', 'min_child_weight'):
    if k_ in best_params: best_params[k_] = int(best_params[k_])
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
FEAT_LO_FULL, FEAT_HI_FULL = X_train.min(0), X_train.max(0); FEAT_LO, FEAT_HI = FEAT_LO_FULL, FEAT_HI_FULL
def burst_ids(y, plc, ts): return pd.DataFrame({'y': y, 'p': plc, 't': ts}).groupby(['y', 'p', 't'], sort=False).ngroup().to_numpy()
bid_te = burst_ids(y_test, plc_te, ts_te)
atk_te = y_test != 0; ben_te = ~atk_te

names = np.array(FEATURE_COLS)
is_cy = np.isin(np.arange(N_FEAT), CYBER_IDX)
x2 = np.array([n.startswith('cyber_x2_') for n in names]); x5 = np.array([n.startswith('cyber_x5_') for n in names]); legacy = is_cy & ~x2 & ~x5
DERIVED = ['voltage_imbalance', 'current_imbalance', 'power_factor', 'freq_deviation']
der = np.array([n in DERIVED for n in names])
print(f'cyber {is_cy.sum()} = legacy {legacy.sum()} + 2s-window {x2.sum()} + 5s-window {x5.sum()} | physical {(~is_cy).sum()} of which derived {der.sum()}')
assert der.sum() == 4 and (legacy.sum() + x2.sum() + x5.sum()) == is_cy.sum(), 'feature groups do not match the expected structure'
SETS = {'phys (33)': ~is_cy, 'cyber (60)': is_cy, 'legacy cyber': legacy, 'legacy+2s cyber': legacy | x2, 'phys+legacy': ~is_cy | legacy,
        'all (93)': np.ones(N_FEAT, bool), 'all - derived': ~der}
SETS = {k: np.flatnonzero(v) for k, v in SETS.items()}
for k, v in SETS.items(): print(f'  {k:16s} {len(v):3d} features, of which cyber {int(is_cy[v].sum())}')
REF = 'all (93)'
rng0 = np.random.RandomState(0)
wb_sel = np.concatenate([rng0.choice(np.where(y_test == c)[0], size=min(WB_PER_CLASS, int((y_test == c).sum())), replace=False) for c in range(1, 6)])
print(f'attack sample: {len(wb_sel)} rows ({WB_PER_CLASS}/class) from {len(np.unique(bid_te[wb_sel]))} distinct bursts | test {len(y_test):,} rows, {len(np.unique(bid_te))} bursts')

## 2. Attack, model and gate helpers (NB3c / NB5b white-box code, NB3 PGD, NB6 gate)

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def pgd(model, X, y, eps=PGD_EPS, n_iter=PGD_ITERS, bs=65536):
    model.eval(); alpha = eps / 4; out = np.empty_like(X)
    for s in range(0, len(X), bs):
        yt = torch.from_numpy(y[s:s + bs]).long().to(DEVICE); X0 = torch.from_numpy(X[s:s + bs]).float().to(DEVICE); Xa = X0.clone()
        for _ in range(n_iter):
            Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]; Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
        out[s:s + bs] = Xa.detach().cpu().numpy()
    return out
def fit_pair(X, y, seed):
    # GPU-fitted classifier (for the white-box margin search) and a CPU copy (the oracle for every reported number, as in NB4)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X, y, verbose=False)
    tmp = f'_tmp_nb11_{seed}_{os.getpid()}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu
print('helpers defined')

In [ ]:
def aps_scores(P, U):
    # randomised APS: score of label y = (mass of labels with higher probability) + U * p_y ; one U per row
    order = np.argsort(-P, 1); Ps = np.take_along_axis(P, order, 1); cum = np.cumsum(Ps, 1)
    S_sorted = cum - (1 - U)[:, None] * Ps; S = np.empty_like(P); np.put_along_axis(S, order, S_sorted, 1); return S
def scores_of(P, method, U): return aps_scores(P, U) if method == 'aps' else 1.0 - P
def qhat(s, alpha):
    n = len(s); k = int(math.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else float(np.sort(s)[k - 1])
def calibrate(Pc, yc, Uc, method, alpha, bid_c=None):
    # returns a scalar threshold, or a per-class vector for lac_cc; lac_dedup keeps one row per burst
    if method == 'lac_cc': return np.array([qhat(1 - Pc[yc == c, c], alpha) if (yc == c).any() else np.inf for c in range(6)])
    if method == 'lac_dedup':
        keep = np.unique(bid_c, return_index=True)[1]; return qhat(1 - Pc[keep, yc[keep]], alpha)
    S = scores_of(Pc, method, Uc); return qhat(S[np.arange(len(yc)), yc], alpha)
def predict_sets(P, method, q, U):
    S = scores_of(P, 'aps' if method == 'aps' else 'lac', U)
    return S <= (q[None, :] if np.ndim(q) else q)
def eval_sets(sets, y, bid, boot_seed=0):
    n = len(y); covered = sets[np.arange(n), y]; size = sets.sum(1); out = {}
    out['cov'] = float(covered.mean()); out['size'] = float(size.mean()); out['single'] = float((size == 1).mean()); out['empty'] = float((size == 0).mean())
    out['cov_pc'] = np.array([covered[y == c].mean() if (y == c).any() else np.nan for c in range(6)]); out['size_pc'] = np.array([size[y == c].mean() if (y == c).any() else np.nan for c in range(6)])
    out['worst_cov'] = float(np.nanmin(out['cov_pc'][np.bincount(y, minlength=6) > 0]))
    out['only_benign'] = float(((size == 1) & sets[:, 0]).mean()); out['has_benign'] = float(sets[:, 0].mean())
    u, inv = np.unique(bid, return_inverse=True); nb = len(u); cb = np.bincount(inv, covered.astype(float), nb); nr = np.bincount(inv, minlength=nb).astype(float)
    idx = np.random.RandomState(boot_seed).randint(nb, size=(N_BOOT, nb)); cs = cb[idx].sum(1) / nr[idx].sum(1)
    out['cov_lo'], out['cov_hi'] = [float(v) for v in np.percentile(cs, [2.5, 97.5])]; out['n_bursts'] = nb
    p = out['cov']; out['cov_row_hw'] = float(1.96 * math.sqrt(max(p * (1 - p), 1e-12) / n))
    return out
def exch_control(Pc, yc, bid_c, alpha, rng, reps=20):
    ub = np.unique(bid_c); covs = []
    for _ in range(reps):
        A = set(rng.permutation(ub)[:len(ub) // 2].tolist()); inA = np.fromiter((b in A for b in bid_c), bool, len(bid_c))
        q = qhat(1 - Pc[inA, yc[inA]], alpha); covs.append(float(((1 - Pc[~inA, yc[~inA]]) <= q).mean()))
    return float(np.mean(covs))
def conf_by_burst(y, pred, bid):
    u, inv = np.unique(bid, return_inverse=True); return np.bincount(inv * 36 + y * 6 + pred, minlength=len(u) * 36).reshape(len(u), 6, 6).astype(float)
def f1_from_conf(C):
    tp = np.einsum('...ii->...i', C); rs = C.sum(-1); cs = C.sum(-2)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where(rs + cs > 0, 2 * tp / (rs + cs), 0.0); rec = np.where(rs > 0, tp / rs, np.nan)
    sup = rs > 0; macro = (f1 * sup).sum(-1) / sup.sum(-1); weighted = (f1 * rs).sum(-1) / rs.sum(-1)
    return macro, weighted, f1, rec
def ece_of(P, y, bins=N_BINS):
    conf = P.max(1); acc = (P.argmax(1) == y); b = np.minimum((conf * bins).astype(int), bins - 1); e = 0.0; rows = []
    for k in range(bins):
        m = b == k
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean()); rows.append((k, int(m.sum()), float(conf[m].mean()), float(acc[m].mean())))
    return float(e), rows
def brier(P, y): return float(((P - np.eye(6)[y]) ** 2).sum(1).mean())
def nll(P, y): return float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1)).mean())
def temp_scale(P, T): L = np.log(np.clip(P, 1e-12, 1)) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def auroc(neg, pos):
    if len(neg) == 0 or len(pos) == 0: return float('nan')
    return float(roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[neg, pos]))
print('conformal machinery defined')

In [ ]:
def load_clf(path):
    # GPU copy for the white-box margin search, CPU copy as the oracle for every reported number (as in NB4 / NB11)
    gpu = xgb.XGBClassifier(); gpu.load_model(path)
    if 'device' in BACKEND: gpu.set_params(device='cuda')
    cpu = xgb.XGBClassifier(); cpu.load_model(path); cpu.set_params(device='cpu', n_jobs=-1); return gpu, cpu
def load_gate(seed):
    try: ck = torch.load(f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE, weights_only=False)
    except TypeError: ck = torch.load(f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE)
    ms_ = []
    for st in ck['state']:
        m = AE(N_FEAT).to(DEVICE); m.load_state_dict(st); m.eval(); ms_.append(m)
    return ms_, float(ck['tau'])
@torch.no_grad()
def gate_score(ms_, X, bs=65536):
    X = np.ascontiguousarray(X, dtype=np.float32); out = np.empty(len(X), np.float64)
    for s in range(0, len(X), bs):
        xb = torch.from_numpy(X[s:s + bs]).to(DEVICE); out[s:s + bs] = torch.stack([((m(xb) - xb) ** 2).mean(1) for m in ms_]).mean(0).double().cpu().numpy()
    return out
def pc_rate(flag, y):  # per attack class share of rows with flag True (classes 1..5)
    return np.array([flag[y == c].mean() if (y == c).any() else np.nan for c in range(1, 6)])
print('helpers defined')

## 3. Per-seed run

For each seed: 7 feature-set models are fitted and attacked (block A); then the base model (the 93-feature one), the robust and the defended model are attacked and combined with the gate (block B).

In [ ]:
RES = {}; t_all = time.time(); y_s = y_test[wb_sel]
for seed in SEEDS:
    pf = f'ablation_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([best_params, seed, list(SETS), WB_PER_CLASS, WB_BUDGETS, PGD_EPS, float(X_train[:5].sum()), SMOKE_TEST, 'nb14-v1'], sort_keys=True, default=str)
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: RES[seed] = saved['res']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); A = {}; B = {}
    base_pair = None
    for nm, idx in SETS.items():
        ts_ = time.time(); gpu, cpu = fit_pair(X_train[:, idx], y_train, seed); Xte = X_test[:, idx]
        P = cpu.predict_proba(Xte); pred = P.argmax(1); C = conf_by_burst(y_test, pred, bid_te); mac, wei, f1c, rec = f1_from_conf(C.sum(0))
        a = {'n_feat': len(idx), 'macro': float(mac), 'weighted': float(wei), 'f1_pc': f1c, 'recall_pc': rec, 'fpr': float((pred[ben_te] != 0).mean()), 'atk': {}}
        FEAT_LO, FEAT_HI = FEAT_LO_FULL[idx], FEAT_HI_FULL[idx]                        # the attack reads these globals
        TH = get_thresholds(gpu); mg = Margin(gpu); X0 = Xte[wb_sel]; clean_ev = (pred[wb_sel] == 0)
        plan = []
        cyb_pos = np.flatnonzero(is_cy[idx])
        if len(cyb_pos): plan += [('cyber', b, cyb_pos) for b in WB_BUDGETS]
        plan += [('any', WB_BUDGETS[0], np.arange(len(idx)))]
        for atk_name, B_, feats in plan:
            xa = whitebox_trace(mg, X0, y_s, feats, TH, B_, verbose=False)[0]; pa = cpu.predict(xa); ev = (pa == 0)
            a['atk'][(atk_name, B_)] = {'evasion': float(ev.mean()), 'evasion_pc': pc_rate(ev, y_s), 'clean_evasion': float(clean_ev.mean()), 'clean_evasion_pc': pc_rate(clean_ev, y_s)}
        FEAT_LO, FEAT_HI = FEAT_LO_FULL, FEAT_HI_FULL
        A[nm] = a
        print(f'  [{nm:16s}] {len(idx):3d} feat | F1 {wei:.4f} (macro {mac:.4f}) | ' + ' | '.join(f'{k[0]} {k[1]:g}: {v["evasion"]:.3f}' for k, v in a['atk'].items()) + f' | clean miss {clean_ev.mean():.3f} | {(time.time() - ts_) / 60:.1f} min', flush=True)
        if nm == REF: base_pair = (gpu, cpu)
        else: del gpu
    # ---------------- block B: classifier x gate
    sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE)); sur.eval()
    X_pgd = pgd(sur, X_test[wb_sel], y_s); ms_, tau = load_gate(seed)
    sc_clean_all = gate_score(ms_, X_test); fl_clean_all = sc_clean_all > tau
    models = {'base': base_pair, 'rob': load_clf(f'xgb_robust_{PIPELINE_VERSION}_seed{seed}.json'), 'dfn': load_clf(f'xgb_defended_{PIPELINE_VERSION}_seed{seed}.json')}
    B['gate_tau'] = tau; B['gate_fpr'] = float(fl_clean_all[ben_te].mean()); B['gate_det_clean'] = float(fl_clean_all[atk_te].mean())
    for mn, (gpu, cpu) in models.items():
        t1 = time.time(); Pf = cpu.predict_proba(X_test); pf_ = Pf.argmax(1); C = conf_by_burst(y_test, pf_, bid_te); mac, wei, f1c, rec = f1_from_conf(C.sum(0))
        B[(mn, 'clean_perf')] = {'weighted': float(wei), 'macro': float(mac), 'f1_pc': f1c}
        B[(mn, 'benign')] = {'fa_cls': float((pf_[ben_te] != 0).mean()), 'fa_gate': float(fl_clean_all[ben_te].mean()), 'fa_both': float(((pf_ != 0) | fl_clean_all)[ben_te].mean())}
        FEAT_LO, FEAT_HI = FEAT_LO_FULL, FEAT_HI_FULL; TH = get_thresholds(gpu); mg = Margin(gpu)
        conds = {'clean': X_test[wb_sel], 'pgd': X_pgd}
        for b_ in WB_BUDGETS: conds[f'wb{b_:g}'] = whitebox_trace(mg, X_test[wb_sel], y_s, CYBER_IDX, TH, b_, verbose=False)[0]
        for cn, Xc in conds.items():
            pr = cpu.predict(Xc); fl = gate_score(ms_, Xc) > tau; e_c = pr == 0; e_g = ~fl; e_b = e_c & e_g
            B[(mn, cn)] = {'ev_cls': float(e_c.mean()), 'ev_gate': float(e_g.mean()), 'ev_both': float(e_b.mean()), 'ev_cls_pc': pc_rate(e_c, y_s), 'ev_gate_pc': pc_rate(e_g, y_s), 'ev_both_pc': pc_rate(e_b, y_s)}
        print(f'  [{mn:4s}] F1 {wei:.4f} | false alarms cls {B[(mn, "benign")]["fa_cls"]:.3f} +gate {B[(mn, "benign")]["fa_both"]:.3f} | evasion cls->both: ' + ' | '.join(f'{cn} {B[(mn, cn)]["ev_cls"]:.3f}->{B[(mn, cn)]["ev_both"]:.3f}' for cn in conds) + f' | {(time.time() - t1) / 60:.1f} min', flush=True)
    r = {'A': A, 'B': B}; RES[seed] = r; joblib.dump({'fp': cfg_fp, 'res': r}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min | gate: threshold {tau:.5f}, benign false alarm {B["gate_fpr"]:.3f}, clean attack detection {B["gate_det_clean"]:.3f}', flush=True)
print(f'all seeds done in {(time.time() - t_all) / 60:.1f} min')

## 4. Statistics helpers

In [ ]:
def ptest(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (float('nan') if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
def dz(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); s = d.std(ddof=1) if len(d) > 1 else 0
    return float(d.mean() / s) if s > 0 else float('nan')
def one_sample_p(x, mu):
    x = np.asarray(x, float)
    if len(x) < 2 or x.std(ddof=1) == 0: return float('nan')
    return float(stats.ttest_1samp(x, mu).pvalue)
def fp(p): return '< 0.0001' if (p == p and p < 5e-5) else (f'{p:.4f}' if p == p else 'n/a')
ms = lambda x, d=3: f'{np.mean(x):.{d}f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.{d}f}'
V = lambda key: np.array([RES[s][key] for s in SEEDS]); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); T = {}
_dz0 = dz
def dz(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    return float('nan') if (len(d) < 2 or d.std(ddof=1) < 1e-9) else float(d.mean() / d.std(ddof=1))
_p0 = ptest
def ptest(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    return float('nan') if (len(d) < 2 or d.std(ddof=1) < 1e-9) else _p0(a, b)

## 5. A. Feature ablation

Paired over seeds, each set against the 93-feature model, Holm-corrected within each table. Seed std is model variance only.

In [ ]:
print('5a. Clean performance (tuned XGBoost per feature set), mean +/- std over seeds. p = paired t-test vs the 93-feature model, Holm over the sets.')
rows = []; pl = []
for nm in SETS:
    w = np.array([RES[s]['A'][nm]['weighted'] for s in SEEDS]); m_ = np.array([RES[s]['A'][nm]['macro'] for s in SEEDS]); fpr = np.array([RES[s]['A'][nm]['fpr'] for s in SEEDS])
    w0 = np.array([RES[s]['A'][REF]['weighted'] for s in SEEDS])
    rows.append({'feature set': nm, 'n features': RES[SEEDS[0]]['A'][nm]['n_feat'], 'weighted F1': ms(w, 4), 'macro F1': ms(m_, 4), 'benign false-alarm': ms(fpr, 4),
                 'Δ weighted vs all': f'{np.mean(w - w0):+.4f}' if nm != REF else '--', 'dz': f'{dz(w, w0):+.2f}' if nm != REF else '--'}); pl.append(ptest(w, w0) if nm != REF else np.nan)
ph = holm(np.array([p if p == p else 1.0 for p in pl]))
for i, nm in enumerate(SETS): rows[i]['p (Holm)'] = fp(ph[i]) if nm != REF else '--'
TA = {}; TA['clean'] = pd.DataFrame(rows); print(TA['clean'].to_string(index=False))
print('\n5b. Per-class recall by feature set (mean over seeds)')
rec = pd.DataFrame({nm: np.nanmean([RES[s]['A'][nm]['recall_pc'] for s in SEEDS], 0) for nm in SETS}, index=class_names).T.round(3); TA['recall_pc'] = rec.reset_index().rename(columns={'index': 'feature set'}); print(rec.to_string())
print('\n5c. White-box tree-greedy attack against each set\'s own trees. Evasion = share of attack rows classified benign after the attack (clean = already missed before it).')
rows = []; keys = sorted({k for s in SEEDS for nm in SETS for k in RES[s]['A'][nm]['atk']}, key=lambda k: (k[0] != 'cyber', k[0], k[1]))
for nm in SETS:
    for k in keys:
        if k not in RES[SEEDS[0]]['A'][nm]['atk']: continue
        ev = np.array([RES[s]['A'][nm]['atk'][k]['evasion'] for s in SEEDS]); ce = np.array([RES[s]['A'][nm]['atk'][k]['clean_evasion'] for s in SEEDS])
        ref_ev = np.array([RES[s]['A'][REF]['atk'][k]['evasion'] for s in SEEDS]) if k in RES[SEEDS[0]]['A'][REF]['atk'] else None
        rows.append({'feature set': nm, 'attacker': k[0], 'budget (sigma)': k[1], 'clean miss': ms(ce, 3), 'evasion after attack': ms(ev, 3), 'increase': ms(ev - ce, 3),
                     'p vs all (paired)': fp(ptest(ev, ref_ev)) if (ref_ev is not None and nm != REF) else '--'})
TA['attack'] = pd.DataFrame(rows); print(TA['attack'].to_string(index=False))
print('\n5d. Evasion after the cyber attacker (0.3 sigma) per attack class, mean over seeds')
pcd = {nm: np.nanmean([RES[s]['A'][nm]['atk'][('cyber', 0.3)]['evasion_pc'] for s in SEEDS], 0) for nm in SETS if ('cyber', 0.3) in RES[SEEDS[0]]['A'][nm]['atk']}
pcd = pd.DataFrame(pcd, index=atk_names).T.round(3); TA['attack_pc'] = pcd.reset_index().rename(columns={'index': 'feature set'}); print(pcd.to_string())

## 6. B. Defence components: classifier x gate

Evasion is measured on the same attack rows in every column. "cls" = the classifier alone, "gate" = the gate alone, "both" = the row is classed benign **and** not flagged. The attack targets the classifier's trees only.

In [ ]:
CONDS = ['clean', 'pgd'] + [f'wb{b:g}' for b in WB_BUDGETS]; MODELS = ['base', 'rob', 'dfn']; TB = {}
Bv = lambda mn, cn, k: np.array([RES[s]['B'][(mn, cn)][k] for s in SEEDS])
print(f'6a. Gate on its own: threshold = 95th percentile of NB6 out-of-fold benign error. Benign false-alarm {ms(np.array([RES[s]["B"]["gate_fpr"] for s in SEEDS]), 4)}; clean attack-row detection {ms(np.array([RES[s]["B"]["gate_det_clean"] for s in SEEDS]), 4)}')
print('\n6b. Clean performance and false alarms')
rows = []
for mn in MODELS:
    w = np.array([RES[s]['B'][(mn, 'clean_perf')]['weighted'] for s in SEEDS]); fa = lambda k: np.array([RES[s]['B'][(mn, 'benign')][k] for s in SEEDS])
    rows.append({'classifier': mn, 'weighted F1': ms(w, 4), 'false alarm: classifier': ms(fa('fa_cls'), 4), 'false alarm: gate': ms(fa('fa_gate'), 4), 'false alarm: classifier OR gate': ms(fa('fa_both'), 4)})
TB['clean'] = pd.DataFrame(rows); print(TB['clean'].to_string(index=False))
print('\n6c. Evasion of attack rows (mean +/- std over seeds): classifier alone | gate alone | both | observed / independence product')
rows = []
for cn in CONDS:
    for mn in MODELS:
        c_, g_, b_ = Bv(mn, cn, 'ev_cls'), Bv(mn, cn, 'ev_gate'), Bv(mn, cn, 'ev_both'); prod = c_ * g_
        rows.append({'condition': cn, 'classifier': mn, 'cls alone': ms(c_, 3), 'gate alone': ms(g_, 3), 'both': ms(b_, 3), 'both / (cls x gate)': f'{np.mean(b_) / np.mean(prod):.2f}' if np.mean(prod) > 0 else 'n/a'})
TB['evasion'] = pd.DataFrame(rows); print(TB['evasion'].to_string(index=False))
print('\n6d. Paired tests on the evasion, Holm within each condition (4 contrasts): does white-box adversarial training (rob) or PGD/PPO adversarial training (dfn) help? does the gate help on top of it? is the pair better than the gate alone?')
rows = []
for cn in CONDS:
    ct = [('rob vs base (classifier alone)', Bv('rob', cn, 'ev_cls'), Bv('base', cn, 'ev_cls')), ('dfn vs base (classifier alone)', Bv('dfn', cn, 'ev_cls'), Bv('base', cn, 'ev_cls')),
          ('base + gate vs base', Bv('base', cn, 'ev_both'), Bv('base', cn, 'ev_cls')), ('rob + gate vs rob', Bv('rob', cn, 'ev_both'), Bv('rob', cn, 'ev_cls')),
          ('rob + gate vs gate alone', Bv('rob', cn, 'ev_both'), Bv('rob', cn, 'ev_gate'))]
    ps = [ptest(a, b) for _, a, b in ct]; ph = holm(np.array([p if p == p else 1.0 for p in ps]))
    for i, (lab, a, b) in enumerate(ct): rows.append({'condition': cn, 'contrast': lab, 'mean difference': f'{np.mean(a - b):+.3f}', 'dz': f'{dz(a, b):+.2f}', 'p (Holm)': fp(ph[i])})
TB['paired'] = pd.DataFrame(rows); print(TB['paired'].to_string(index=False))
print('\n6e. Evasion by attack class under the 1.0-sigma white-box attack (mean over seeds): classifier alone vs with gate')
cn = f'wb{WB_BUDGETS[-1]:g}'; pcr = {}
for mn in MODELS:
    pcr[f'{mn} cls'] = np.nanmean([RES[s]['B'][(mn, cn)]['ev_cls_pc'] for s in SEEDS], 0); pcr[f'{mn} +gate'] = np.nanmean([RES[s]['B'][(mn, cn)]['ev_both_pc'] for s in SEEDS], 0)
pcr['gate alone'] = np.nanmean([RES[s]['B'][('base', cn)]['ev_gate_pc'] for s in SEEDS], 0)
pcr = pd.DataFrame(pcr, index=atk_names).T.round(3); TB['evasion_pc'] = pcr.reset_index().rename(columns={'index': 'configuration'}); print(pcr.to_string())
print('\nReading guide: the gate is the same object in every row; its evasion differs by column only because the attacked rows differ. A "both / (cls x gate)" ratio near 1 means the two layers fail independently; above 1 they fail on the same rows (less protection than a product suggests); below 1, on different rows.')

## 7. Checks, figure and saved outputs

In [ ]:
chk = []
w_all = np.array([RES[s]['A'][REF]['weighted'] for s in SEEDS]); w_base = np.array([RES[s]['B'][('base', 'clean_perf')]['weighted'] for s in SEEDS])
chk.append({'check': 'block A 93-feature model and block B base model are the same fit', 'value': f'{np.max(np.abs(w_all - w_base)):.2e}', 'ok': bool(np.max(np.abs(w_all - w_base)) < 1e-9)})
if not SMOKE_TEST: chk.append({'check': 'clean weighted F1 of the 93-feature model close to the NB2 value 0.8635', 'value': f'{w_all.mean():.4f}', 'ok': bool(abs(w_all.mean() - 0.8635) < 0.01)})
chk.append({'check': 'cyber attacker evasion >= clean miss for every set (attack never helps the defender)', 'value': 'see 5c', 'ok': bool(all(RES[s]['A'][nm]['atk'][k]['evasion'] >= RES[s]['A'][nm]['atk'][k]['clean_evasion'] - 1e-9 for s in SEEDS for nm in SETS for k in RES[s]['A'][nm]['atk']))})
chk.append({'check': 'both <= min(cls, gate) in every cell', 'value': 'see 6c', 'ok': bool(all(RES[s]['B'][(mn, cn)]['ev_both'] <= min(RES[s]['B'][(mn, cn)]['ev_cls'], RES[s]['B'][(mn, cn)]['ev_gate']) + 1e-9 for s in SEEDS for mn in MODELS for cn in CONDS))})
chk.append({'check': 'attack sample identical in every set and condition', 'value': f'{len(wb_sel)} rows', 'ok': True})
TC = pd.DataFrame(chk); print(TC.to_string(index=False)); assert TC['ok'].all(), 'a check failed'

fig, ax = plt.subplots(2, 2, figsize=(16, 10)); nms = list(SETS); xx = np.arange(len(nms))
ax[0, 0].bar(xx, [np.mean([RES[s]['A'][n]['weighted'] for s in SEEDS]) for n in nms], yerr=[np.std([RES[s]['A'][n]['weighted'] for s in SEEDS], ddof=1) for n in nms], color='C0', capsize=3)
ax[0, 0].set_xticks(xx); ax[0, 0].set_xticklabels(nms, rotation=30, ha='right', fontsize=8); ax[0, 0].set_ylim(0.4, 0.95); ax[0, 0].set_title('A. Clean weighted F1 by feature set'); ax[0, 0].grid(alpha=.3, axis='y')
for j, (k, col) in enumerate([(('cyber', 0.3), 'C1'), (('cyber', 1.0), 'C3'), (('any', 0.3), 'C2')]):
    vals = [np.mean([RES[s]['A'][n]['atk'][k]['evasion'] for s in SEEDS]) if k in RES[SEEDS[0]]['A'][n]['atk'] else np.nan for n in nms]
    ax[0, 1].bar(xx + (j - 1) * 0.27, vals, 0.27, color=col, label=f'{k[0]} attacker, {k[1]:g} sigma')
ax[0, 1].set_xticks(xx); ax[0, 1].set_xticklabels(nms, rotation=30, ha='right', fontsize=8); ax[0, 1].set_title('A. Evasion after white-box attack (empty = attacker has no features to move)'); ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=.3, axis='y')
w_ = 0.12; cfgs = [('base', 'ev_cls'), ('base', 'ev_both'), ('rob', 'ev_cls'), ('rob', 'ev_both'), ('dfn', 'ev_cls'), ('dfn', 'ev_both')]
for j, (mn, k) in enumerate(cfgs):
    ax[1, 0].bar(np.arange(len(CONDS)) + (j - 2.5) * w_, [Bv(mn, cn, k).mean() for cn in CONDS], w_, yerr=[Bv(mn, cn, k).std(ddof=1) for cn in CONDS], capsize=2, color=f'C{j}', label=f'{mn} {"+ gate" if k == "ev_both" else "alone"}')
ax[1, 0].plot([], []); ax[1, 0].set_xticks(range(len(CONDS))); ax[1, 0].set_xticklabels(CONDS); ax[1, 0].set_title('B. Evasion of attack rows'); ax[1, 0].legend(fontsize=7, ncol=2); ax[1, 0].grid(alpha=.3, axis='y')
for mn, mk in zip(MODELS, ['o', 's', '^']):
    for k, fill in [('cls', 'C0'), ('both', 'C3')]:
        fa = np.mean([RES[s]['B'][(mn, 'benign')][f'fa_{k}'] for s in SEEDS]); ev = Bv(mn, CONDS[-1], 'ev_cls' if k == 'cls' else 'ev_both').mean()
        ax[1, 1].scatter(fa, ev, marker=mk, s=90, color=fill, label=f'{mn} {"+ gate" if k == "both" else "alone"}')
ax[1, 1].set_xlabel('benign false-alarm rate (clean test)'); ax[1, 1].set_ylabel(f'evasion under {CONDS[-1]}'); ax[1, 1].set_title('B. Cost / benefit of the gate'); ax[1, 1].legend(fontsize=7); ax[1, 1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'ablation_{PIPELINE_VERSION}.png', dpi=150); plt.show()
OUT = {f'A_{k}': v for k, v in TA.items()}; OUT.update({f'B_{k}': v for k, v in TB.items()}); OUT['checks'] = TC
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': OUT, 'per_seed': {s: RES[s] for s in SEEDS}, 'config': {'WB_PER_CLASS': WB_PER_CLASS, 'WB_BUDGETS': WB_BUDGETS, 'sets': {k: v.tolist() for k, v in SETS.items()}}}, f'ablation_results_{PIPELINE_VERSION}.pkl')
for k_, t_ in OUT.items(): t_.to_csv(f'ablation_{k_}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB14', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb14_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=1)
for f in [f'ablation_results_{PIPELINE_VERSION}.pkl', f'ablation_{PIPELINE_VERSION}.png', f'nb14_run_manifest_{PIPELINE_VERSION}.json'] + [f'ablation_{k_}_{PIPELINE_VERSION}.csv' for k_ in OUT]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)